In [ ]:
import logging  #bilgi vermeyi sağlayan standart kütüphane bilgileri print ile vermektense logging içindeki farklı durumlara göre çıktı vermek daha doğru bir kullanımdır
from pathlib import Path
import torch #modelin sinir ağı bu kütüphane üzerinden çalışacak
import cv2
from transformers import AutoProcessor, AutoModelForMultimodalLM
#Processor image i modelin anlayabileceği sayısal verilere(çok boyutlu sayısal diziler) çevirir
#AutoModelForMultimodalLM → MiniCPM-V modelini yükler config dosyasına göre modeli yükler
#AutoProcessor            → Görsel + metni modele hazırlar

In [ ]:
#modelin ve processorun ortama yüklenmesi 
MODEL_ID = "openbmb/MiniCPM-V-4.6-BNB"

processor = AutoProcessor.from_pretrained(MODEL_ID) #Sadece görsel ve metni ileride nasıl hazırlayacağını bilen processor nesnesini oluşturuyor.
model = AutoModelForMultimodalLM.from_pretrained(
        MODEL_ID,
        device_map="auto" #modelin nerede çalıştırılacağı otomatik belirlenecek (CPU veya GPU)
    )
model.eval() #eğitim değil de inference modunda çalıştırılacak. Katmanların çalışma davranışını inference'a uygun hale getirir.

In [ ]:
#modelin kurulumu durumu hakkında bilgilendirme
print("Model device:")
print(model.device)

print("Model class")
print(type(model))

total_params = sum(parameter.numel() for parameter in model.parameters())
print(f"Total parameters: {total_params}")

In [ ]:
#image in yüklenmesi ve sorunun belirlenmesi
IMAGE_PATH = Path("image.jpg")

image_brg = cv2.imread(str(IMAGE_PATH))
if image_brg is None:
    raise ValueError(f"Image not found at {IMAGE_PATH}")
image_rgb = cv2.cvtColor(image_brg, cv2.COLOR_BGR2RGB) #renk kanallarını değiştiriyoruz

question = "Where is the cat in the image?"

In [ ]:
#chat modeli için inputu belirli bir konuşma yapısına getirilmesi
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": image_rgb
            },
            {
                "type": "text",
                "text": question
            }
        ]
    }
]

In [ ]:
#mesajın processor a verilmesi ve sonucunda inputun artık modelin anlayacağı bir formata dönüştürülmesi
inputs = processor.apply_chat_template(
    messages,
    tokenize=True,
    add_generate_prompt=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = inputs.to(model.device)

In [ ]:
#modelin cevap üretmesi
with torch.inference_mode(): #inference modunda olduğumuz için gradient hesaplarını kapatıyor
    outputs = model.generate(
        **inputs,
        max_new_tokens=100, #cevabın uzunluğu 100 token ile sınırlandırılıyor
    )

generated_tokens = outputs[:, inputs["input_ids"].shape[1]:] #model çıktı üretirken çıkışın başında inputu da eklediği için onu kesip sadece modelin ürettiği kısmı alıyoruz
#burada inputs["input_ids"] kısmı girdilerin dictionarysindeki girdi değerleirni alır
#.shade[1] ile de tensorün boyutunu gösterir
# : ile de tensör boyutu kadar olan kısımdaki tensörleri keser.

#çıktı oluşturulurken girdinin hemen arkasına yeni tokenler ekleniyor o yüzden cevabı oluşturuken promptu silmek gerekli

In [ ]:
#modelin ürettiği cevabı tokenlerden stringe çevirme
response = processor.batch_decode(
    generated_tokens,
    skip_special_tokens=True,
)[0]

print("Modelin ham cevabı:")
print(response)